# Extra lab X1 · Advanced prompt engineering, measured

**Time:** 55 min · **Work in teams of 3** · **An extra lab: use it if there is spare time, after Lab 16**

**Where we are in the day:** Lab 16 taught the basic prompt practices (a role and task, a fixed format, marking the user's text, hard examples, a way out) with five prompts, P1 to P5. Its best prompt, P4, reached 70.8 per cent exact with 464 tokens in our test run, short of the target of 90, and Lab 17 then trained the model. This extra lab starts from that best prompt and asks what the more advanced techniques add, and what each costs.
**Today's question:** of the advanced prompt techniques, which ones pay for themselves on a small model, and how do we know without trusting our feelings?
**What you will do:**
- Run the same 72 golden messages through about forty prompts, and for each technique write a **prediction**, read a **measurement**, and make a **decision**.
- Techniques: roles and delimiters, hard examples against easy ones, a grammar that forces valid JSON, voting, reasoning, negative examples and an "unsure" answer, prompt injection, and an automatic prompt search.
- Keep every prompt with its score, and write a regression check that stops a "small tidy-up" from breaking the assistant.

**What you will see:** a scoreboard with exact %, prompt tokens and time for every prompt, and a decision log with one line per technique.
**Why it matters on a device:** a prompt costs tokens on every request, and a "clever" technique that adds model calls or tokens must earn its cost in accuracy, safety or reliability.

**How to run this lab**
- Click **Kernel, Restart Kernel and Run All Cells**. Write your guess on paper before each cell marked **Predict**.
- All the cells together take about **4 minutes** on a GPU pod once the models are on the disk (section X1.4 runs on the CPU and takes about 2 minutes of that).
- If a cell takes much longer, interrupt the kernel and ask the trainer. Do not start it again and again.
- Before you start: Lab 16 must have been run. Its best prompt (P4) is our starting point, and X1 reads its saved scoreboard row. **Lab 18 must have been run for section X1.4**, because it makes the GGUF files used there. If the files are missing, that section says so and the rest of the lab still works.
- Close the notebooks of other labs first: two notebooks holding the model in the GPU memory may not fit.

**Which model this lab uses.** All sections except X1.4 use the **base** Qwen2.5-1.5B-Instruct, the same model as Lab 16, with **no adapter**. We use the base model on purpose: a fine-tuned model has learned one prompt and ignores the rest, so prompt work shows its effect on the base model. Section X1.4 (the grammar) uses the **fine-tuned and quantised** GGUF files of Lab 18, because that is the model that goes on the device.

**Words used in this lab** (each is explained again where it first matters):
- **Role:** who is speaking in a chat prompt: `system` (standing instructions), `user` (the message), `assistant` (an earlier answer, used for examples).
- **Delimiter:** a marker around the user's text (quotes, or tags such as `<message>`) so the model can tell our instructions from the person's words.
- **Few-shot examples:** worked examples (a message and its right answer) put inside the prompt.
- **Grammar (GBNF):** a short description of exactly which texts are allowed. With it, the model can only write texts that match.
- **Self-consistency:** ask the same question several times with randomness and take the most common answer (a vote).
- **Prompt injection:** a message that tries to change the assistant's instructions instead of being a normal message.
- **Development half and confirmation half:** we choose a prompt on one half of the test messages and check it on the other half, which the choice never saw.
- **Overfitting:** a prompt that looks best only because we chose it on those very messages.
- **Regression check:** a test that runs again after every change and says whether something got worse.

**The rule of the day (the same as Lab 16): change one thing, run the same test, write the numbers down.** One message of the 72 is 1.4 points, so a difference under **3 points** (two messages) is noise. The code prints this verdict for you.

**Python in this lab:** loops, lists, dictionaries and `print`. A prompt is described by one dictionary, called a **spec**. The code that runs a spec and scores it is in `x_prompting.py`; open it if you are curious.

In [ ]:
import sys, os
for folder in ["..", "../.."]:                  # find the course folder (the one that holds slm_lab.py)
    if os.path.exists(os.path.join(folder, "slm_lab.py")):
        sys.path.insert(0, os.path.abspath(folder))
        break
import pandas as pd
import edge_lab as el
import edge_nlu as nlu
import x_prompting as xp
from slm_lab import check, FAST
tok, model = el.load_edge_model()
golden = nlu.load_golden()
print("model:", el.MODEL_ID, "(the BASE model, no adapter)")
print("weights:", el.model_size_mb(model), "MB | golden messages:", len(golden))

**You should see** the model name and about 3,000 MB of weights, as in Lab 16, and 72 golden messages.

## X1.0 The ground rules
**The test set and the choice.** When we choose between prompts using the golden set, the golden set slowly stops being a fair exam: we have chosen what suits it. So we split it, kind by kind, into a **development half** (we choose on it) and a **confirmation half** (we only look at it afterwards). We keep one list of 36 positions for each half.

**The examples are never test messages.** Every example we put in a prompt was written by us. The next cell checks this by code: a message that is both an example and a test message would make a prompt look better than it is.

In [ ]:
dev_idx, confirm_idx = xp.split_dev_confirm(golden)
print("development half:", len(dev_idx), "messages | confirmation half:", len(confirm_idx), "messages")
all_shots = xp.SHOTS_LAB16 + xp.HARD_POOL + xp.SHOTS_EASY8 + xp.NEAR_MISS_SHOTS + xp.UNSURE_SHOTS
print("examples that are also test messages:", xp.leaked_examples(all_shots, golden))
board = []                                       # the scoreboard of this lab: one line for each prompt we measure
decisions = []                                   # the decision log: one line for each technique

In [ ]:
check(len(dev_idx) == 36 and len(confirm_idx) == 36 and xp.leaked_examples(all_shots, golden) == [],
      "Checkpoint: two halves of 36 messages, and no example is a test message.",
      "The halves are not 36 and 36, or an example is also a test message. Do not edit data/edge_golden.json or x_prompting.py.")

**You should see** an empty list of leaked examples. (The old version of Lab 16 used the example "start movie night", which is also a test message. Always check this by code, as here.)

Every prompt we try goes through one small function, so that the **only** thing that changes is the prompt. It runs the spec on the 72 messages, scores it, and adds one line to the scoreboard.

In [ ]:
def measure(spec):
    row = xp.run_spec(tok, model, spec, golden)
    board.append(xp.board_line(row))
    return row

## X1.1 The baseline: the best prompt of Lab 16
We start exactly where Lab 16 stopped: its **P4**, the role, task, format and quotes sentence (P3's system message) plus its **eight examples**, with the message in double quotes. Lab 16 already taught these practices; here we measure what comes after them. This is our "before" row. **Predict:** write the exact % you expect (Lab 16 saved its best row on the scoreboard; the next cells print it).

In [ ]:
SPEC_BASE = xp.make_spec("A Lab 16 best (P4): 8 examples, quotes", xp.SYSTEM_L16_P3, xp.SHOTS_LAB16)
base_row = measure(SPEC_BASE)
print(base_row["name"], "| exact:", base_row["exact %"], "% | prompt tokens:", base_row["prompt tokens"], "| ms per message:", base_row["ms per message"])
print("by kind:", base_row["by_kind"])

**You should see** a score close to the row Lab 16 saved (in our test run P4 gave 70.8 per cent with 464 prompt tokens), and a weak spot in some kinds of message.

In [ ]:
saved = el.load_scoreboard().get("1 prompt only")
if saved is None:
    print("Lab 16 was not run on this pod, so there is no row to compare with.")
else:
    print("Lab 16 saved:", saved["what"], "|", saved["exact_percent"], "% with", saved["prompt_tokens"], "tokens")
    check(abs(saved["exact_percent"] - base_row["exact %"]) <= 4, "Checkpoint: this baseline is close to the row Lab 16 saved.",
          "The numbers differ by more than 4 points. Lab 16's best prompt may not have been P4, or the model differs. Ask the trainer.")

## X1.2 Roles and delimiters
A chat prompt has **roles**. The `system` message holds standing instructions, `user` holds the message, and earlier `assistant` messages can show examples. A model is trained on this layout, so it should follow it better than a pile of text. A **delimiter** (quotes, or tags such as `<message> ... </message>`) shows where the person's words begin and end.

Lab 16 marked the user's text with quotes and one sentence. We try four changes against the baseline (A): everything in one user message with no system message (B); the examples written as text inside the system message (C); no delimiter at all (D); and tags instead of quotes (E). For D and E the sentence about the quotes is changed to fit (no sentence for D, a sentence about the tags for E).

**Predict.** Write on paper which of B, C, D, E will score **lower** than A, and which will be about the same. (Remember: under 3 points is noise.)

In [ ]:
specs = [
    xp.make_spec("B one block, no system message", xp.SYSTEM_L16_P3, xp.SHOTS_LAB16, layout="one_block"),
    xp.make_spec("C examples inside the system message", xp.SYSTEM_L16_P3, xp.SHOTS_LAB16, layout="system_text"),
    xp.make_spec("D no delimiter", xp.system_for_wrap("plain"), xp.SHOTS_LAB16, wrap="plain"),
    xp.make_spec("E tags", xp.system_for_wrap("tags"), xp.SHOTS_LAB16, wrap="tags"),
]
role_rows = [base_row]
for spec in specs:
    role_rows.append(measure(spec))

In [ ]:
table = []
for row in role_rows:
    line = xp.board_line(row)
    line["against A"] = xp.verdict(row["exact %"] - base_row["exact %"])
    table.append(line)
pd.DataFrame(table)

**You should see** five rows and a verdict for each. The layout without a system message (B) gets the chat template's own default system message, so it is not free of roles either.

**Decision rule of this lab:** choose the **cheapest prompt that is within 2 points of the best** (fewest prompt tokens). It protects us from paying tokens for a gain that is only noise.

In [ ]:
chosen = xp.pick_best(role_rows, 2.0)
WRAP = chosen["spec"]["wrap"]
LAYOUT = chosen["spec"]["layout"]
xp.add_decision(decisions, "roles and delimiters", chosen["name"],
                str(chosen["exact %"]) + " % exact with " + str(chosen["prompt tokens"]) + " prompt tokens")

In [ ]:
check(len(role_rows) == 5, "Checkpoint: five layouts measured (A to E).",
      "The table should have 5 rows. Run the cells of X1.2 again, in order.")

## X1.3 Choosing the examples
Examples cost tokens on every request, so choose them with care. We test three things, each with the same prompt apart from the examples:
1. **Hard against easy.** Lab 16's eight examples were already chosen as hard cases. We compare them with eight **easy** examples (clean messages with clear answers) and with eight of our own **hard** examples (we wrote 16: noisy speech-recogniser text, Hinglish, a bare value, a command for another device, a missing slot).
2. **Order.** The same eight hard examples in the opposite order. A model often pays most attention to the last example.
3. **How many.** 4, 8, 12 and 16 hard examples.

**Predict.** Write: (1) will hard examples beat easy examples, and by how many points? (2) will the order matter, yes or no? (3) after how many examples do you expect the gain to stop?

In [ ]:
SYSTEM = xp.system_for_wrap(WRAP)
hard8 = xp.HARD_POOL[:8]
lab_row = measure(xp.make_spec("Lab 16 eight", SYSTEM, xp.SHOTS_LAB16, WRAP, LAYOUT))
easy_row = measure(xp.make_spec("easy 8", SYSTEM, xp.SHOTS_EASY8, WRAP, LAYOUT))
hard_row = measure(xp.make_spec("hard 8", SYSTEM, hard8, WRAP, LAYOUT))
print("Lab 16 eight:", lab_row["exact %"], "% |", lab_row["prompt tokens"], "tokens")
print("easy 8      :", easy_row["exact %"], "% |", easy_row["prompt tokens"], "tokens")
print("hard 8      :", hard_row["exact %"], "% |", hard_row["prompt tokens"], "tokens ->", xp.verdict(hard_row["exact %"] - easy_row["exact %"]), "against easy 8")

In [ ]:
kinds = list(easy_row["by_kind"].keys())
table = []
for kind in kinds:
    table.append({"kind": kind, "easy 8": easy_row["by_kind"][kind], "Lab 16 eight": lab_row["by_kind"][kind], "hard 8": hard_row["by_kind"][kind]})
pd.DataFrame(table)

**You should see** where the hard examples help or do not help, kind by kind. Hard examples should help the kinds they resemble (noisy, Hinglish, short, out of domain).

Now the order of the examples.

In [ ]:
reversed8 = []
for example in hard8:
    reversed8.insert(0, example)                  # put each example in front: the list is now the other way round
order_row = measure(xp.make_spec("hard 8, reversed order", SYSTEM, reversed8, WRAP, LAYOUT))
print("hard 8        :", hard_row["exact %"], "%")
print("hard 8 reversed:", order_row["exact %"], "% ->", xp.verdict(order_row["exact %"] - hard_row["exact %"]))

**You should see** a verdict. If it says "within noise", the order does not matter for this model and this task, and we should not spend time on it.

Now how many examples.

In [ ]:
hard_by_count = {8: hard_row}
for count in [4, 12, 16]:
    hard_by_count[count] = measure(xp.make_spec("hard " + str(count), SYSTEM, xp.HARD_POOL[:count], WRAP, LAYOUT))
curve = []
previous = None
for count in [4, 8, 12, 16]:
    row = hard_by_count[count]
    line = {"examples": count, "exact %": row["exact %"], "prompt tokens": row["prompt tokens"]}
    if previous is not None:
        line["gain in points"] = round(row["exact %"] - previous["exact %"], 1)
        line["extra tokens"] = row["prompt tokens"] - previous["prompt tokens"]
    curve.append(line)
    previous = row
pd.DataFrame(curve)

**You should see** the gain per step and the tokens it costs. Look for the step where the gain becomes noise while the tokens keep growing: that is where to stop.

We apply the decision rule from X1.2 to all the candidates of this section.

In [ ]:
candidates = [lab_row, easy_row, hard_by_count[4], hard_by_count[8], hard_by_count[12], hard_by_count[16]]
cur_row = xp.pick_best(candidates, 2.0)
CUR = cur_row["spec"]
xp.add_decision(decisions, "examples", cur_row["name"],
                str(cur_row["exact %"]) + " % exact with " + str(cur_row["prompt tokens"]) + " prompt tokens (easy 8 gave " + str(easy_row["exact %"]) + " %)")

In [ ]:
check(len(hard_by_count) == 4 and CUR is not None, "Checkpoint: the examples are chosen and the choice is in the decision log.",
      "Run the cells of X1.3 again, in order. If one cell stopped with an error, read the last line of the error.")

## X1.4 A grammar that forces valid JSON
So far we **asked** the model to write JSON and hoped. With a **grammar** we do not ask: at every step the engine removes the tokens that would break the grammar, so the model can only write an allowed text. The grammar language of llama.cpp is called **GBNF**. *In one sentence: a grammar makes valid JSON a guarantee instead of a hope.*

Here is the grammar of our answer. Read it from the top: `root` is the whole answer; `intent` can only be one of nine words; the slots can only be the four we know, with only the values we allow.

In [ ]:
print(xp.GBNF_ANSWER)

This part runs on the **CPU** with the **fine-tuned GGUF files of Lab 18**. We test two files on the same 18 messages (every fourth of the 72), with and without the grammar: `Q4_K_M` (the file we would ship) and `Q2_K` (the small file that lost accuracy and sometimes wrote invalid JSON in Lab 18). The prompt is the short one the model was trained with.

First the lab checks that the files exist.

In [ ]:
ready, message = xp.gguf_status(["Q4_K_M", "Q2_K"])
print(message)

In [ ]:
check(ready, "Checkpoint: the GGUF files of Lab 18 and llama-cpp-python are ready.",
      "Run Lab 18 first (it makes the GGUF files), then run this section again. If you skip this section, the rest of the lab still works.")

**You should see** a message that the files were found. If you see the warning instead, the cells of this section print "skipped" and the lab goes on.

**Predict.** Write on paper: (1) will the grammar change the **exact %** of `Q4_K_M`? (2) will it raise the **valid JSON %** of `Q2_K` to 100? (3) will it change the **speed**, and in which direction? In Lab 18, `Q4_K_M` was at 83.3 per cent exact with 100 per cent valid JSON, and `Q2_K` at 45.8 per cent exact with 87.5 per cent valid JSON.

In [ ]:
import edge_gguf as eg
gguf_rows = []
sample = xp.every_nth(golden, 4)

def test_file(level):
    path = os.path.join(xp.GGUF_FOLDER, "intent-" + level + ".gguf")
    eg.warm_up(path)
    llm = eg.load_gguf(path)
    for mode in ["no grammar", "with grammar"]:
        grammar = None
        if mode == "with grammar":
            grammar = xp.GBNF_ANSWER
        result = xp.evaluate_gguf_grammar(llm, nlu.SHORT_SYSTEM, sample, grammar, max_seconds=40)
        gguf_rows.append(xp.gguf_table_row(level, mode, result))

Now run it for the two files. It takes about two minutes.

In [ ]:
if ready:
    for level in ["Q4_K_M", "Q2_K"]:
        test_file(level)
else:
    print("skipped: the GGUF files are not ready")

In [ ]:
if ready:
    print(pd.DataFrame(gguf_rows))

**You should see** four rows. **Valid JSON %** is about the **syntax**: can code parse the reply? **Bad answers** is about the **content**: the reply parses, but code still cannot act on it, because it has an unknown intent, a slot or value that is not allowed, or a temperature outside 16 to 30. With the grammar the valid JSON must be 100 per cent for both files. The grammar also lists the allowed intents, rooms, states, scenes and temperatures, so bad answers should drop to 0; if some remain, read them in the table. **The lesson: a grammar guarantees the SYNTAX, and the values it allows, but never that the answer is RIGHT.** In our pod run the small file `Q2_K` went from 5.6 to 16.7 per cent exact with the grammar: valid JSON every time, but mostly the wrong meaning. `Q4_K_M` reached 94.4 per cent exact with the grammar.

One more test, to see the grammar override the prompt. We give the model a prompt that asks for a sentence in plain English, once without and once with the grammar.

In [ ]:
if ready:
    path = os.path.join(xp.GGUF_FOLDER, "intent-Q4_K_M.gguf")
    llm = eg.load_gguf(path)
    odd_system = "Say in one short sentence what the person wants."
    for text in ["turn on the kitchen light", "set the bedroom ac to 24", "what is the weather today"]:
        plain, seconds = xp.ask_gguf_grammar(llm, odd_system, text)
        forced, seconds = xp.ask_gguf_grammar(llm, odd_system, text, xp.GBNF_ANSWER)
        print(text, "\n   without:", plain, "\n   with   :", forced)
    del llm

**You should see** that without the grammar the model follows the odd prompt, and with it the reply is still the JSON shape. So the grammar can replace the format instructions in the prompt (fewer tokens), and a bad prompt cannot break the format. **The risk:** the grammar forces an answer even when the model is lost, so it can hide doubt. Lab 20's confidence check is still needed.

In [ ]:
if ready:
    with_grammar = []
    for row in gguf_rows:
        if row["decoding"] == "with grammar":
            with_grammar.append(row)
    check(len(with_grammar) == 2 and with_grammar[0]["valid JSON %"] == 100.0 and with_grammar[1]["valid JSON %"] == 100.0,
          "Checkpoint: with the grammar every reply is valid JSON, for both files.",
          "A file gave invalid JSON even with the grammar. Check that the grammar printed above is unchanged, and ask the trainer.")

Now the decision. We compare the two files, with and without the grammar, from the numbers of the table.

In [ ]:
if ready:
    q2_plain = gguf_rows[2]
    q2_forced = gguf_rows[3]
    evidence = "Q2_K valid JSON " + str(q2_plain["valid JSON %"]) + " to " + str(q2_forced["valid JSON %"]) + " %, exact " + str(q2_plain["exact %"]) + " to " + str(q2_forced["exact %"]) + " %"
    xp.add_decision(decisions, "grammar", "use a grammar when code must read the answer; keep the confidence check", evidence)

## X1.5 Self-consistency: a vote against its cost
The vote is a technique Lab 16 no longer covers, so we do it properly here. For each of 36 messages (every second one) we ask for **5 answers with randomness** (temperature 0.7) and take the most common **meaning** (intent and slots together). We can then read the result for a vote of 1, 3 and 5 answers from the same samples. The work grows with the number of answers, so we write the **relative work** next to the score.

**Predict.** Write: will a vote of 3 beat the plain greedy answer? By how many points? Will a vote of 5 beat a vote of 3?

In [ ]:
half = xp.every_nth(golden, 2)
greedy = xp.run_spec(tok, model, CUR, half)
prompt_texts = []
for case in half:
    prompt_texts.append(xp.make_prompt(tok, CUR, case["text"]))
out = xp.sample_answers(tok, model, prompt_texts, how_many=5, temperature=0.7)
print("greedy on", len(half), "messages:", greedy["exact %"], "% exact,", greedy["answer tokens"], "answer tokens per message")
print("5 sampled answers each took", round(out["seconds"], 1), "seconds and", out["answer_tokens"], "answer tokens in total")

In [ ]:
vote_table = [{"way": "greedy (one answer, no randomness)", "exact %": greedy["exact %"], "relative work": 1}]
for k in [1, 3, 5]:
    vote_table.append({"way": "sampled, vote of " + str(k), "exact %": xp.vote_scores(out["answers"], half, k), "relative work": k})
pd.DataFrame(vote_table)

**You should see** that a vote of 1 (one random answer) is usually not better than greedy, and that the work grows in step with the number of answers. Whether the vote pays is a number you can read here.

In [ ]:
best_vote = vote_table[1]
for line in vote_table[1:]:
    if line["exact %"] > best_vote["exact %"]:
        best_vote = line
gain = best_vote["exact %"] - greedy["exact %"]
print("best vote:", best_vote["way"], "->", xp.verdict(gain), "against greedy, for", best_vote["relative work"], "times the work")
if gain >= xp.NOISE_POINTS:
    xp.add_decision(decisions, "vote", "vote only on the messages the model is unsure about", "gain " + str(round(gain, 1)) + " points for " + str(best_vote["relative work"]) + " times the work")
else:
    xp.add_decision(decisions, "vote", "stay with greedy decoding", "the best vote was " + xp.verdict(gain))

## X1.6 Reasoning prompts against direct answers
"Think step by step" can help on hard problems. Our answer is one short label, so reasoning may only cost time. We try two ways of asking for a reason, each with **examples that show a reason too** (a model copies its examples): a line `REASON: ...` before the JSON, and a `"reason"` field inside the JSON. The reasons in the examples are short sentences made from the intent.

**Predict.** Write: will either reasoning prompt score higher than the direct answer? How many times slower will each be (1.5, 3 or 6)?

In [ ]:
LINE_RULE = "First write one short line starting with REASON: saying why, then the JSON on the next line."
FIELD_RULE = 'Add a first field "reason" with one short sentence: {"reason": "...", "intent": "...", "slots": {...}}'
shots_line = xp.shots_with_reason(CUR["shots"], "line")
shots_field = xp.shots_with_reason(CUR["shots"], "field")
spec_line = xp.make_spec("reason line first", CUR["system"] + "\n" + LINE_RULE, shots_line, CUR["wrap"], CUR["layout"], 120)
spec_field = xp.make_spec("reason field in the JSON", CUR["system"] + "\n" + FIELD_RULE, shots_field, CUR["wrap"], CUR["layout"], 120)
line_row = measure(spec_line)
field_row = measure(spec_field)
print("direct:", cur_row["exact %"], "% |", cur_row["answer tokens"], "answer tokens |", cur_row["ms per message"], "ms")

In [ ]:
table = []
for row in [cur_row, line_row, field_row]:
    line = xp.board_line(row)
    line["against direct"] = xp.verdict(row["exact %"] - cur_row["exact %"])
    line["time vs direct"] = str(round(row["ms per message"] / cur_row["ms per message"], 1)) + " times"
    table.append(line)
pd.DataFrame(table)

**You should see** the answer tokens and the time grow for both reasoning prompts, and a verdict on the accuracy. (In the old version of Lab 16, "think first" without examples scored lower than the format-only prompt, 41.7 against 47.2 per cent.)

In [ ]:
best_reason = line_row
if field_row["exact %"] > line_row["exact %"]:
    best_reason = field_row
gain = best_reason["exact %"] - cur_row["exact %"]
if gain >= xp.NOISE_POINTS:
    xp.add_decision(decisions, "reasoning", best_reason["name"] + " pays for its time", xp.verdict(gain) + ", " + str(best_reason["ms per message"]) + " ms against " + str(cur_row["ms per message"]))
else:
    xp.add_decision(decisions, "reasoning", "answer directly", "the best reasoning prompt was " + xp.verdict(gain) + " and slower")

## X1.7 Negative examples and an "unsure" option
Examples usually show what to do. A **negative example** shows a message that *looks* like a job but is not, with the answer `other`. A **wrong and right pair** tells the model about a mistake we have seen. The third idea is different: give the model a way out. If it may answer `unsure`, a bad guess can become a question back to the user (the idea of Lab 20, now done in the prompt).

Lab 16's P5 already sent doubtful messages to `other`. We try three more ideas on top of the current prompt: (F) three near-miss examples (a television, a phone screen, a timer); (G) the wrong and right pair about inventing a room; (H) the `unsure` option, with two examples. An `unsure` reply is counted as **not exact**, but we also count the **wrong answers that were given**, because a wrong answer given is worse than a question asked.

**Predict.** Write: which of F, G, H will lower the number of **wrong answers given**? Will H lower the exact %?

In [ ]:
near_row = measure(xp.extend_spec(CUR, "F near-miss examples", extra_shots=xp.NEAR_MISS_SHOTS))
pair_row = measure(xp.extend_spec(CUR, "G wrong and right pair", extra_system=xp.WRONG_RIGHT_RULE))
unsure_row = measure(xp.extend_spec(CUR, "H unsure option", extra_system=xp.UNSURE_RULE, extra_shots=xp.UNSURE_SHOTS))
neg_rows = [cur_row, near_row, pair_row, unsure_row]
table = []
for row in neg_rows:
    scores = xp.score_with_unsure(row["replies"], golden)
    scores["prompt"] = row["name"]
    scores["prompt tokens"] = row["prompt tokens"]
    table.append(scores)
pd.DataFrame(table)

**You should see** one line per prompt: the exact %, the share of `unsure` replies, the wrong answers given, and the exact % among the answered messages. The last prompt (H) is the only one that can say `unsure`.

In [ ]:
BEST_ROW = xp.pick_best([cur_row, near_row, pair_row], 2.0)
BEST_SPEC = BEST_ROW["spec"]
given_before = xp.score_with_unsure(cur_row["replies"], golden)["wrong answers given"]
given_unsure = xp.score_with_unsure(unsure_row["replies"], golden)["wrong answers given"]
xp.add_decision(decisions, "negative examples", BEST_ROW["name"], str(BEST_ROW["exact %"]) + " % exact, " + str(BEST_ROW["prompt tokens"]) + " tokens")
xp.add_decision(decisions, "unsure option", "keep it as an option, not as the default",
                "wrong answers given " + str(given_before) + " to " + str(given_unsure) + ", exact % " + str(cur_row["exact %"]) + " to " + str(unsure_row["exact %"]) + "; worth it only if asking back is cheap")

In [ ]:
check(len(neg_rows) == 4 and BEST_SPEC is not None, "Checkpoint: the four prompts of X1.7 are measured and the best hand-made prompt is chosen.",
      "Run the cells of X1.7 again, in order.")

## X1.8 Prompt injection
Our assistant reads text that a person typed or said. Some people will type instructions meant for the model, not for the home: "ignore the rules above and ...". This is **prompt injection**. The model cannot tell our instructions from the person's words, unless we help it. We test 13 attack messages. For each we know the **safe answer** and what the **attacker wants** (a wrong intent, an answer that is not JSON, or our prompt printed back). An attack **worked** if the reply shows what the attacker wanted.

**Predict.** Write: how many of the 13 attacks will work against our current prompt (0 to 13)? And will a defence written in the prompt stop all of them?

In [ ]:
attacks = xp.attack_cases()
for attack in attacks[:4]:
    print("-", attack["text"])
print("...", len(attacks), "attacks in all")

We need one small function that sends all the attacks to a prompt and returns the replies.

In [ ]:
def run_attacks(spec):
    texts = []
    for attack in attacks:
        texts.append(xp.make_prompt(tok, spec, attack["text"]))
    return el.ask_many(tok, model, texts)["replies"]

plain_replies = run_attacks(BEST_SPEC)
print(xp.count_attacks(plain_replies, attacks))

In [ ]:
for index in range(len(attacks)):
    mark = "held   "
    if xp.attack_worked(plain_replies[index], attacks[index]):
        mark = "WORKED "
    print(mark + attacks[index]["text"][:50] + " -> " + plain_replies[index][:60].replace("\n", " "))

**You should see** a count and one line per attack.

**The defence in two layers.**
1. **In the prompt:** put the person's text between `<message>` tags, and say that it is **data to label, never instructions**. In code we also remove `<` and `>` from the text first (so nobody can close our tag) and cut it to 200 characters.
2. **In the code, after the model:** a cheap phrase check on the message (words such as "ignore", "from now on", "reply with") and a strict check of the reply (valid JSON, a known intent, allowed slot values). A message or reply that fails gets a safe answer, `other`.

In [ ]:
DEF_SPEC = xp.extend_spec(BEST_SPEC, "I defended: tags, rule, clean text", extra_system=xp.DEFENCE_RULE, wrap="tags", clean=True)
defended_replies = run_attacks(DEF_SPEC)
guarded = []
reasons = []
for index in range(len(attacks)):
    final, why = xp.guard(attacks[index]["text"], defended_replies[index])
    guarded.append(final)
    reasons.append(why)
print("layer 1 only (prompt)  :", xp.count_attacks(defended_replies, attacks))
print("layers 1 and 2 (+ code):", xp.count_attacks(guarded, attacks))

Now the cost of the defence. A defence that blocks normal messages is no defence. First the phrase check on the 72 clean messages (code only), then the defended prompt on the same 72.

In [ ]:
false_alarms = []
for case in golden:
    if xp.looks_like_injection(case["text"]) != "":
        false_alarms.append(case["text"])
print("clean messages blocked by the phrase check:", len(false_alarms), false_alarms)
def_row = measure(DEF_SPEC)
print("defended prompt on the clean messages:", def_row["exact %"], "% against", BEST_ROW["exact %"], "% before ->", xp.verdict(def_row["exact %"] - BEST_ROW["exact %"]))

In [ ]:
summary = []
for name, replies in [("no defence", plain_replies), ("prompt defence", defended_replies), ("prompt and code", guarded)]:
    line = xp.count_attacks(replies, attacks)
    line["defence"] = name
    summary.append(line)
print(pd.DataFrame(summary))
print("which attacks still worked after both layers:")
for index in range(len(attacks)):
    if xp.attack_worked(guarded[index], attacks[index]):
        print("  -", attacks[index]["text"][:70])

**You should see** the table of the three levels, the number of false alarms on clean messages, and the attacks that still work. The last three attacks were written in other words, on purpose, so that the phrase check does not catch them.

**The lesson is about design, not about wording.** A list of phrases can always be walked around, and a model can be talked round. What we can control is **what an injection can do**. Here the model only chooses a *label* from a short list, and code decides what happens: so the worst case is a wrong label, not a free-form action. Risky actions (the lock, "everything off") need a confirmation step in code, as in Day 2. **Never give the model a power you would not give to a stranger who can type to it.**

In [ ]:
worked_both = xp.count_attacks(guarded, attacks)["attacks that worked"]
worked_none = xp.count_attacks(plain_replies, attacks)["attacks that worked"]
xp.add_decision(decisions, "injection", "tags and a data rule in the prompt, checks in code, risky actions behind confirmation",
                "attacks that worked: " + str(worked_none) + " with no defence, " + str(worked_both) + " with both layers; clean messages blocked: " + str(len(false_alarms)))

## X1.9 Automatic prompt search, and why it overfits
Instead of writing prompts by hand we can **generate** them. The helper mixes three role sentences, two format sentences, three sets of rules, three sets of examples and two ways of wrapping the text. That makes 108 prompts; we draw a random sample of them with a fixed seed (6 in a fast run, otherwise 10) and run each on all 72 messages.

We **choose** on the development half and **check** on the confirmation half.

**Predict.** Write: will the prompt that is best on the development half also be the best on the confirmation half? Will its confirmation score be higher, the same or lower than its development score?

In [ ]:
SEARCH_COUNT = 10
if FAST:
    SEARCH_COUNT = 6
variants = xp.make_search_variants(SEARCH_COUNT, seed=5)
for spec in variants:
    print(spec["name"])

In [ ]:
search_rows = []
for spec in variants:
    search_rows.append(xp.run_spec(tok, model, spec, golden))
table = []
for row in search_rows:
    table.append({"variant": row["name"], "dev %": xp.percent_on(row["ok"], dev_idx), "confirm %": xp.percent_on(row["ok"], confirm_idx),
                  "all 72 %": row["exact %"], "prompt tokens": row["prompt tokens"]})
pd.DataFrame(table)

In [ ]:
check(len(search_rows) == SEARCH_COUNT, "Checkpoint: every variant was run on the 72 messages.",
      "The number of rows is not " + str(SEARCH_COUNT) + ". Run the cell above again.")

Now the winner on the development half, and what happens on the confirmation half. For comparison we also give the numbers of our hand-made prompt from X1.7.

In [ ]:
w = xp.best_index(search_rows, dev_idx)
win = search_rows[w]
print("best on the development half:", win["name"])
print("  development:", xp.percent_on(win["ok"], dev_idx), "% | confirmation:", xp.percent_on(win["ok"], confirm_idx), "% | tokens:", win["prompt tokens"])
print("hand-made   :", xp.percent_on(BEST_ROW["ok"], dev_idx), "% | confirmation:", xp.percent_on(BEST_ROW["ok"], confirm_idx), "% | tokens:", BEST_ROW["prompt tokens"])
confirm_best = xp.best_index(search_rows, confirm_idx)
print("best on the confirmation half would have been:", search_rows[confirm_best]["name"])

**You should see** the winner's score on the half it was chosen on and on the half it was not. If the confirmation score is lower, we **overfitted**: we chose the prompt that suited those 36 messages. One run is a story; many runs are a measurement, so the next cell repeats the choice for many random splits (code only, no model). It does it twice: with 36 messages to choose on, and with only 12.

In [ ]:
splits = []
for size in [36, 12]:
    result = xp.many_splits(search_rows, 72, size, how_many=30)
    result["messages to choose on"] = size
    splits.append(result)
pd.DataFrame(splits)

**You should see** two rows. Read them like this: the winner is always better on the part it was chosen on than on the rest; the gap is the **overfitting**, and it is larger when we choose on fewer messages. The winner is worth choosing only if its score on the rest is clearly above the average of all prompts on the rest.

In [ ]:
gap = xp.percent_on(win["ok"], dev_idx) - xp.percent_on(win["ok"], confirm_idx)
xp.add_decision(decisions, "prompt search", "trust the confirmation score, not the development score",
                "the winner lost " + str(round(gap, 1)) + " points on the half it had not seen; the average gap over 30 splits of 12 is " + str(round(splits[1]["winner on dev part"] - splits[1]["same winner on the rest"], 1)))

**One more warning.** After this section the 72 golden messages have been **used for choosing**, so they are no longer a clean exam for the final prompt. A real project keeps a **third set** of messages that nobody has tuned on, and looks at it once, at the end. Lab 16.9 (your own five messages) is a very small start; one message there is 20 points.

## X1.10 Your turn: add one rule, and judge it on both halves
Write **one sentence** to add to the prompt: a rule about a kind of message that is still wrong (look at `BEST_ROW["wrong"]` if you wish). The starter measures your rule and prints the score on each half, next to the hand-made prompt. A rule that helps on the development half and not on the confirmation half is overfitting. If you are stuck, open the solution cell below the starter.

In [ ]:
MY_RULE = ""                                    # TODO: write one rule sentence, in plain words
my_row = measure(xp.extend_spec(BEST_SPEC, "my rule", extra_system=MY_RULE))
print("development half  :", xp.percent_on(BEST_ROW["ok"], dev_idx), "% before ->", xp.percent_on(my_row["ok"], dev_idx), "% with your rule")
print("confirmation half :", xp.percent_on(BEST_ROW["ok"], confirm_idx), "% before ->", xp.percent_on(my_row["ok"], confirm_idx), "% with your rule")

In [ ]:
change = my_row["exact %"] - BEST_ROW["exact %"]
check(MY_RULE.strip() != "", "Checkpoint: you wrote a rule and measured it.", "MY_RULE is empty. Write one rule sentence in the starter cell and run it again.")
print("exact % with your rule against the prompt without it:", xp.verdict(change))
if change < -6:
    print("Your rule hurt by more than 6 points. A rule can also hurt: that is why we measure. Print one wrong reply with  print(my_row['wrong'][0])  and read it.")
elif change > 6:
    print("Your rule helped by more than 6 points. Look at the confirmation half above: did it help there too?")
else:
    print("Your rule changed the score by less than 6 points. A rule can also hurt or do nothing: that is why we measure.")

## X1.11 Prompt versions and a regression check
A prompt is **code**. It changes, it can get worse, and nobody remembers why. So we keep every version **with its score**, in a file (`outputs/x1_prompt_book.json`). Then a check can compare a new prompt with a saved one before we accept it.

We save four versions: v1 is Lab 16's P3 (role, format and quotes sentence, no examples), v2 the easy examples, v3 our hand-made best prompt, v4 the winner of the automatic search.

In [ ]:
book = xp.load_book()
v1_row = measure(xp.make_spec("Lab 16 P3", xp.SYSTEM_L16_P3))
book = xp.save_version(book, "v1", "Lab 16 P3: role, format, quotes sentence, no examples", v1_row)
book = xp.save_version(book, "v2", "eight easy examples", easy_row)
book = xp.save_version(book, "v3", "hand-made: " + BEST_ROW["name"], BEST_ROW)
book = xp.save_version(book, "v4", "automatic search winner: " + win["name"], win)
table = []
for entry in book:
    table.append({"version": entry["version"], "note": entry["note"], "exact %": entry["exact %"], "prompt tokens": entry["prompt tokens"]})
pd.DataFrame(table)

**You should see** four versions with their scores. Now a **regression check**: someone wants to save tokens and "tidies" v3 by deleting its rules and half its examples. The check has three rules: the exact % must not fall by more than 2 points; **no kind of message** may fall by more than 15 points (a fall in one kind can hide behind an average); and the prompt must fit the 400-token budget of Lab 16.

**Predict.** Write: will the shortened prompt pass all three rules? Which rule is the most likely to catch it?

In [ ]:
saved_v3 = xp.find_version(book, "v3")
short_spec = xp.make_spec("v5 shortened", xp.SYSTEM_L16_P2, BEST_SPEC["shots"][:4], BEST_SPEC["wrap"], BEST_SPEC["layout"])
v5_row = measure(short_spec)
passed = xp.show_report(xp.regression_report(saved_v3, v5_row, budget_tokens=400))
print("all rules passed:", passed)

In [ ]:
table = []
for kind in saved_v3["by_kind"]:
    table.append({"kind": kind, "v3 saved": saved_v3["by_kind"][kind], "v5 shortened": v5_row["by_kind"].get(kind, 0.0)})
pd.DataFrame(table)

**You should see** a PASS or FAIL for each rule, and the table by kind that explains it. If everything passes, the tidy-up was safe on this set; the point is that now we **know** instead of hoping.

In [ ]:
check(len(book) >= 4 and isinstance(passed, bool), "Checkpoint: four versions are saved and the regression check ran.",
      "Run the cells of X1.11 again, in order.")

## X1.12 Your turn: add a rule to the regression check
The check does not look at **time**. A prompt with a long reasoning part can keep the accuracy and still make every answer slower. Write a function `my_rule(saved, row)` that returns `False` when the new prompt's `ms per message` is **more than 1.3 times** the saved version's, and `True` otherwise. The cell then tests it on two pairs of numbers. If you are stuck, open the solution cell below.

In [ ]:
def my_rule(saved, row):
    ok = True                                    # TODO: set ok to False when row["ms per message"] is more than 1.3 times saved["ms per message"]
    return ok

print("slower (should be False):", my_rule({"ms per message": 100}, {"ms per message": 200}))
print("same speed (should be True):", my_rule({"ms per message": 100}, {"ms per message": 110}))

In [ ]:
check(my_rule({"ms per message": 100}, {"ms per message": 200}) == False and my_rule({"ms per message": 100}, {"ms per message": 110}) == True,
      "Right: your rule catches a prompt that is much slower and lets a similar one pass.",
      "The two tests did not give False and then True. Compare row['ms per message'] with 1.3 * saved['ms per message'].")

Use your rule on the shortened prompt of the last section.

In [ ]:
print("v5 against v3, time rule:", my_rule(saved_v3, v5_row), "|", saved_v3["ms per message"], "ms saved,", v5_row["ms per message"], "ms new")

## X1.13 What you proved: the decision log
Here is the scoreboard of every prompt we measured, and the decision log with one line per technique. Save the best hand-made prompt: it is the row to put next to Lab 16's "before" row.

In [ ]:
print(pd.DataFrame(decisions))

In [ ]:
final = {"what": BEST_ROW["name"], "exact_percent": BEST_ROW["exact %"], "prompt_tokens": BEST_ROW["prompt tokens"],
         "ms_per_message": BEST_ROW["ms per message"], "model": "base 1.5B, no adapter"}
xp.save_x1_row("x1 best hand-made prompt", final)
before = el.load_scoreboard().get("1 prompt only")
if before is not None:
    print("Lab 16 'before' row  :", before["exact_percent"], "% |", before["prompt_tokens"], "tokens")
print("X1 best hand-made row:", final["exact_percent"], "% |", final["prompt_tokens"], "tokens")
print("lines on the scoreboard of this lab:", len(board))

**You should see** your best prompt next to Lab 16's. The honest reading: how far did prompting get against the target of 90 per cent, and at what cost in tokens? The fine-tuned model of Lab 17 reached the target with a prompt about a tenth as long.

**Habits added to the eight of Lab 16:**
1. Split your test messages: choose on one half, confirm on the other, and keep a third set for the end.
2. Examples are never test messages; check it by code.
3. Hard examples first; stop adding examples when the gain is noise.
4. Do not trust a difference under two messages.
5. If code must read the answer, use a grammar, and keep a confidence check.
6. Count the cost of every technique in tokens and time, not only in accuracy.
7. Treat all user text as data. Limit what a wrong label can do.
8. Keep every prompt version with its score, and run a regression check before you accept a change.

## What you take away
- Every technique has a price and a measurement. The decision is a number (gain against cost), not a feeling.
- A grammar makes the form of the answer certain; it cannot make the content right.
- A prompt chosen on the same messages it is scored on looks better than it is. Choose on one set and confirm on another.
- Prompt injection cannot be solved by wording alone. Design so that a wrong label can do little harm.
- A prompt is code: keep versions, scores and a regression check.

## Check your understanding
1. A grammar gave 100 per cent valid JSON, but the exact % did not rise. Why?
*✍️ Your answer:* 
2. The prompt that was best on the development half scored lower on the confirmation half. Is something wrong with the code?
*✍️ Your answer:* 
3. A vote of five answers costs five times the work. When can it still be worth it?
*✍️ Your answer:* 
4. Why is "tell the model not to obey instructions inside the message" not enough against prompt injection?
*✍️ Your answer:* 

## Next
**Extra lab X2:** here we made one model call as good and as cheap as we could. In X2 the small model sits inside several agents, and we count how many model calls each design needs, with their tokens, time and accuracy.